# Topic 4: Null Handling

> **Phase 3 → Week 4 → Topic 4**

---

## Why Nulls Are Dangerous in Spark

Nulls silently corrupt results if not handled:

```
data = [100, 200, None, 400]

# Python:         sum([100, 200, None, 400]) → TypeError!
# Pandas:         sum() = 700  (skips None by default)
# Spark avg():    (100+200+400)/3 = 233.3  (ignores nulls)
# Spark count(*): 4  (includes null rows)
# Spark count(col): 3  (excludes null values in that column)

# The Spark trap: count(*) != count(col) when nulls exist!
```

Nulls also affect:
- **Joins**: nulls never match any value (not even other nulls) in join conditions
- **ORDER BY**: nulls appear first in ASC, last in DESC (configurable)
- **Comparisons**: `null == null` is `null`, not `true` — always use `isNull()`
- **Aggregations**: most functions skip nulls silently

---

## Interview Cheat Sheet

**Q: How do you handle nulls in a Spark DataFrame?**
> Four main approaches: (1) `dropna()` — drop rows with nulls in specified columns; (2) `fillna(value)` — replace nulls with a default value; (3) `coalesce(col1, col2)` — use first non-null from a list; (4) `when(col.isNull(), replacement).otherwise(col)` for conditional logic. Choice depends on business context — don't drop if nulls carry meaning.

**Q: How do you check for null in a Spark filter?**
> Use `F.col("name").isNull()` or `F.col("name").isNotNull()`. Never use `== None` or `!= None` — in Spark SQL semantics, `null == null` evaluates to `null` (unknown), not `True`. The `isNull()` method correctly handles SQL null semantics.

**Q: Does count(*) count null rows?**
> Yes. `count(*)` counts all rows including those with nulls. `count(column_name)` counts only non-null values in that column. This difference is a common interview trap and a source of bugs in aggregations.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week4 - Null Handling") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Dataset with deliberate nulls
dirty_data = spark.createDataFrame([
    (1,  "Alice",   "Engineering",  95000,   "India"),
    (2,  "Bob",     None,           88000,   "China"),
    (3,  "Carol",   "Marketing",    None,    "USA"),
    (4,  None,      "Sales",        55000,   None),
    (5,  "Eve",     "Engineering",  None,    "Germany"),
    (6,  "Frank",   None,           72000,   "Japan"),
    (7,  None,      None,           None,    None),
    (8,  "Grace",   "Marketing",    68000,   "Korea"),
], ["id", "name", "dept", "salary", "country"])

print("Dataset with nulls:")
dirty_data.show()

## Part 1: Detecting Nulls

In [ ]:
# isNull / isNotNull
print("Rows where name is NULL:")
dirty_data.filter(F.col("name").isNull()).show()

print("Rows where salary is NOT NULL:")
dirty_data.filter(F.col("salary").isNotNull()).show()

In [ ]:
# Count nulls per column — classic data quality check
null_counts = dirty_data.select([
    F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls")
    for c in dirty_data.columns
])

print("Null counts per column:")
null_counts.show()

# Null percentage
total = dirty_data.count()
null_pct = dirty_data.select([
    F.round(
        F.sum(F.col(c).isNull().cast("int")) / total * 100, 1
    ).alias(f"{c}_null_pct")
    for c in dirty_data.columns
])

print("Null percentage per column:")
null_pct.show()

In [ ]:
# count(*) vs count(col) — the null trap
print("The null trap in count():")
dirty_data.select(
    F.count("*").alias("count_star"),           # counts all rows including nulls
    F.count("name").alias("count_name"),         # counts non-null names
    F.count("salary").alias("count_salary"),     # counts non-null salaries
    F.count("dept").alias("count_dept"),         # counts non-null depts
).show()
print("count(*) = 8, but count(name) = 6 — nulls are silently excluded from column counts!")

## Part 2: Dropping Nulls

In [ ]:
# dropna() — drop rows with any null
print(f"Original: {dirty_data.count()} rows")
print(f"After dropna() [any null]: {dirty_data.dropna().count()} rows")
dirty_data.dropna().show()

# dropna(how='all') — drop only if ALL columns are null
print(f"After dropna(how='all'): {dirty_data.dropna(how='all').count()} rows")

# dropna(subset=[...]) — drop if specific columns are null
print(f"After dropna(subset=['name','salary']): {dirty_data.dropna(subset=['name', 'salary']).count()} rows")
dirty_data.dropna(subset=["name", "salary"]).show()

# dropna(thresh=N) — drop if less than N non-null values in row
print(f"After dropna(thresh=4): {dirty_data.dropna(thresh=4).count()} rows (must have >= 4 non-null cols)")

## Part 3: Filling Nulls

In [ ]:
# fillna() — replace nulls with a value

# Fill all string nulls with 'Unknown'
print("fillna('Unknown') on string columns:")
dirty_data.fillna("Unknown").show()

# Fill numeric nulls with 0
print("fillna(0) on numeric columns:")
dirty_data.fillna(0).show()

In [ ]:
# fillna with dict — different value per column
filled = dirty_data.fillna({
    "name":    "Unknown_Person",
    "dept":    "No_Department",
    "salary":   0,
    "country": "Unknown_Country"
})

print("fillna with per-column defaults:")
filled.show()

# Alternative: using when().otherwise()
print("Same result using when/otherwise (more flexible — can use expressions):")
dirty_data.withColumn(
    "salary_clean",
    F.when(F.col("salary").isNull(), F.lit(0)).otherwise(F.col("salary"))
).select("name", "salary", "salary_clean").show()

In [ ]:
# Fill nulls with a computed value (mean, median)
mean_salary = dirty_data.select(F.avg("salary")).collect()[0][0]
print(f"Mean salary (of non-null rows): {mean_salary:.1f}")

imputed = dirty_data.withColumn(
    "salary_imputed",
    F.coalesce(F.col("salary"), F.lit(mean_salary))
)

print("Salary with mean imputation for nulls:")
imputed.select("name", "salary", "salary_imputed").show()

## Part 4: Nulls in Joins

In [ ]:
# Null keys in joins — nulls NEVER match
left = spark.createDataFrame([
    (1, "Alice"),
    (2, "Bob"),
    (None, "Unknown"),
], ["id", "name"])

right = spark.createDataFrame([
    (1, "Eng"),
    (None, "Sales"),
    (3, "Mkt"),
], ["id", "dept"])

print("INNER JOIN with null keys — nulls never match each other:")
left.join(right, on="id", how="inner").show()
print("Only id=1 matches. null on left never matches null on right!")

print("\nFULL OUTER JOIN — nulls appear but don't connect:")
left.join(right, on="id", how="full").show()

## Part 5: Nulls in Sorting and Aggregation

In [ ]:
# Null ordering in sort
print("orderBy salary ASC — nulls appear FIRST:")
dirty_data.orderBy(F.col("salary").asc_nulls_first()).select("name", "salary").show()

print("orderBy salary ASC — nulls LAST:")
dirty_data.orderBy(F.col("salary").asc_nulls_last()).select("name", "salary").show()

In [ ]:
# Aggregations silently skip nulls
print("Aggregation behavior with nulls:")
dirty_data.select(
    F.avg("salary").alias("avg_skips_null"),       # avg of 95000+88000+55000+72000+68000 / 5
    F.sum("salary").alias("sum_skips_null"),
    F.count("salary").alias("count_non_null"),
    F.count("*").alias("count_all_rows"),
).show()
print("avg uses 5 non-null values, not 8 total rows!")

## Exercises

1. Write a function that prints the null count AND null percentage for every column in a DataFrame.
2. From `dirty_data`, drop rows where BOTH `name` AND `dept` are null (but keep rows with only one of them null).
3. Replace null salaries with the department average salary (not the global average). Hint: `groupBy("dept").agg(avg("salary"))` then join.
4. How many unique departments are there if you treat null as a separate category? Use `fillna` + `countDistinct`.
5. Why does `filter(col("salary") != None)` not work in Spark? What should you use instead?

In [ ]:
# Exercise 1: Null audit function
def null_audit(df):
    total = df.count()
    return df.select([
        F.struct(
            F.sum(F.col(c).isNull().cast("int")).alias("null_count"),
            F.round(F.sum(F.col(c).isNull().cast("int")) / total * 100, 1).alias("null_pct")
        ).alias(c)
        for c in df.columns
    ])

null_audit(dirty_data).show(truncate=False)

In [ ]:
# Exercise 3: Replace null salary with department average
dept_avg = dirty_data.groupBy("dept").agg(F.avg("salary").alias("dept_avg_salary"))

dirty_data.join(dept_avg, on="dept", how="left") \
          .withColumn(
              "salary_filled",
              F.coalesce(F.col("salary"), F.col("dept_avg_salary"))
          ) \
          .select("name", "dept", "salary", "salary_filled") \
          .show()